## Pattern {PATTERN_NAME} Index Building Content

This notebook demonstrates how to process documents and build a vector store index for RAG applications. It covers document discovery, text extraction, chunking, and uploading embeddings to a vector database using Llama Stack.

### &#x1F4CB; Contents 
This notebook contains the following sections:

- **[Setup](#Setup)**
  - [Install packages](#Install-packages)
  - [Import required libraries](#Import-required-libraries)
  - [Configure S3 credentials](#Configure-S3-credentials)
  - [Prepare S3 client](#Prepare-S3-client)
- **[Process input documents](#Process-input-documents)**
  - [Documents discovery](#Documents-discovery)
  - [Text extraction](#Text-extraction)
- **[Upload documents content into vector store](#Upload-documents-content-into-vector-store)**
  - [Prepare Llama Stack Client](#Prepare-Llama-Stack-Client)
  - [Prepare chunker](#Prepare-chunker)
  - [Initialize vector store](#Initialize-vector-store)
  - [Upload chunks to vector store](#Upload-chunks-to-vector-store)
  - [Retrieve chunks for sample question](#Retrieve-chunks-for-sample-question)
- **[Summary](#Summary)**

---

## Setup

This section sets up the notebook environment by installing required packages, importing libraries, and configuring access to S3 storage.

### Install packages

Install all required Python packages for document processing and RAG operations:
- **boto3**: AWS SDK for Python to interact with S3 storage
- **pipelines-components**: Red Hat's pipeline components for data processing
- **docling**: Document processing and text extraction library
- **ai4rag**: The AutoRAG framework for building RAG applications

In [ ]:
%pip install boto3 | tail -n 1
%pip install -U --no-cache-dir git+https://github.com/red-hat-data-services/pipelines-components.git@rhoai-3.4 | tail -n 1
%pip install docling | tail -n 1
%pip install 'ai4rag~=0.5.3' | tail -n 1

### Import required libraries

Import all necessary Python modules and configure logging to suppress verbose output from component loggers.

In [ ]:
import os
import json
import logging
from pathlib import Path
from types import SimpleNamespace
import getpass

import warnings

warnings.filterwarnings("ignore")

import boto3
from langchain_core.documents import Document

for logger_name in (
    "httpx",
    "Document Loader component logger",
    "Text Extraction component logger",
):
    logging.getLogger(logger_name).propagate = False

### Configure S3 credentials

To load documents from S3-compatible object storage, you need to provide credentials. If you're using OpenShift AI, these can be configured as data connections.

&#x1F4CC; **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

&#x1F4A1; **Tip**: In the project, open **Connections** and add an **S3 compatible object storage connection** to a bucket you will use for documents and test data. Open **Workbenches**, edit your workbench, and attach the S3 connection you created so the notebook can read from the bucket. Save and restart the workbench if prompted.

In [ ]:
required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]
missing = [var for var in required_vars if not os.environ.get(var)]
if missing:
    raise ValueError(f"Missing required environment variables: {{missing}}")

### Prepare S3 client

Creates an S3 client session using the provided credentials. This client will be used to discover and download documents from the specified S3 bucket.

In [ ]:
import warnings


def _is_ssl_error(exc: BaseException) -> bool:
    """Check whether an exception (or its cause chain) is an SSL verification failure."""
    seen = set()
    current = exc
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        msg = str(current).upper()
        if "CERTIFICATE_VERIFY_FAILED" in msg or "SSL" in msg:
            return True
        current = current.__cause__ or current.__context__
    return False


def _create_s3_client(verify=True):
    session = boto3.session.Session(
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    )
    return session.client(
        service_name="s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        verify=verify,
    )


try:
    s3_client = _create_s3_client()
    s3_client.list_buckets()
except Exception as exc:
    if _is_ssl_error(exc):
        warnings.warn("SSL verification failed for S3 — retrying with verify=False.")
        s3_client = _create_s3_client(verify=False)
    else:
        raise

---

## Process input documents

This section handles document discovery and text extraction. Documents are first discovered in S3 storage, then their content is extracted and converted to markdown format for further processing.

The data processing pipeline prepares documents for the RAG system in multiple steps. Each step runs as a standalone component with outputs stored under `step_outputs/`. 

| Step | Component | Purpose |
|------|-----------|---------|
| 1 | **Documents discovery** | List documents in the bucket, prioritize benchmark-referenced docs, apply a size cap, and write a JSON manifest (no content download). |
| 2 | **Text extraction** | Download the listed documents from S3 and extract text to Markdown using Docling. |

In [ ]:
from kfp_components.components.data_processing.autorag.documents_discovery.component import documents_discovery
from kfp_components.components.data_processing.autorag.text_extraction.component import text_extraction

step_output_dir = Path("./step_outputs")
input_data_bucket_name = os.environ["AWS_S3_BUCKET"]
input_data_key = "{INPUT_DATA_KEY}"
step_output_dir.mkdir(parents=True, exist_ok=True)

### Documents discovery

Lists objects in the S3 input bucket, filters by supported extensions (e.g., `.pdf`, `.docx`, `.pptx`, `.md`, `.html`, `.txt`), and builds a document set. Documents referenced in the benchmark are prioritized, then others are added until a configurable size limit (1 GB by default) is reached. This step does not download document contents but writes a JSON manifest (`documents_descriptor.json`) containing the bucket, prefix, and list of selected object keys and sizes for the next step.

In [ ]:
discovered_documents_out = SimpleNamespace(path=str(step_output_dir / "discovered_documents"))

documents_discovery.python_func(
    input_data_bucket_name=input_data_bucket_name,
    input_data_path=input_data_key,
    discovered_documents=discovered_documents_out,
)

descriptor_path = step_output_dir / "discovered_documents" / "documents_descriptor.json"
with open(descriptor_path) as f:
    descriptor = json.load(f)

print(json.dumps(descriptor, indent=4, ensure_ascii=False))

### Text extraction

Reads the `documents_descriptor.json` produced by the discovery step, downloads each listed document from S3 into a temporary directory, and runs **Docling** to extract text. Output is one Markdown file per document (e.g., `document_0.md`, `document_1.md`) written to the artifact output path. These files form the final text corpus for the RAG system.

In [ ]:
descriptor_in = SimpleNamespace(path=str(step_output_dir / "discovered_documents"))
extracted_text_out = SimpleNamespace(path=str(step_output_dir / "extracted_text"))

text_extraction.python_func(
    documents_descriptor=descriptor_in,
    extracted_text=extracted_text_out,
)

---

## Upload documents content into vector store

This section configures the vector store, chunks the extracted documents, and uploads embeddings to the database for semantic search.

### Prepare Llama Stack Client

The Llama Stack client provides the interface to the embedding models and vector database. This section initializes the client using API credentials from environment variables or prompts.

**Prerequisites:**
- `LLAMA_STACK_CLIENT_API_KEY`: Your authentication key for the Llama Stack API
- `LLAMA_STACK_CLIENT_BASE_URL`: The base URL of your Llama Stack instance

&#x1F4A1; **Tip**: In OpenShift AI Workbench, you can add these as environment variables or data connections to avoid entering them manually each time.

In [ ]:
from llama_stack_client import LlamaStackClient
import httpx

if not os.getenv("LLAMA_STACK_CLIENT_API_KEY") or not os.getenv("LLAMA_STACK_CLIENT_BASE_URL"):
    os.environ["LLAMA_STACK_CLIENT_API_KEY"] = getpass.getpass("Please enter 'LLAMA_STACK_CLIENT_API_KEY': ")
    os.environ["LLAMA_STACK_CLIENT_BASE_URL"] = getpass.getpass("Please enter 'LLAMA_STACK_CLIENT_BASE_URL': ")

_ls_kwargs = dict(
    base_url=os.getenv("LLAMA_STACK_CLIENT_BASE_URL"),
    api_key=os.getenv("LLAMA_STACK_CLIENT_API_KEY"),
)
if not {LS_SSL_VERIFY}:
    _ls_kwargs["http_client"] = httpx.Client(verify=False)

client = LlamaStackClient(**_ls_kwargs)

### Prepare chunker

The chunker splits extracted documents into smaller chunks for more effective retrieval. Configuration includes:
- **Chunking Method**: The algorithm used to split text (e.g., recursive character splitting)
- **Chunk Size**: Maximum number of characters per chunk
- **Chunk Overlap**: Number of overlapping characters between consecutive chunks to preserve context

Proper chunking ensures that retrieved context is both relevant and fits within the model's context window.

In [ ]:
from ai4rag.rag.chunking import LangChainChunker

chunking_method = "{CHUNKING_METHOD}"
chunk_size = {CHUNK_SIZE}
chunk_overlap = {CHUNK_OVERLAP}

chunker = LangChainChunker(method=chunking_method, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

### Initialize vector store

The vector store manages document embeddings and enables semantic search. This section configures:
- **Embedding Model**: Converts text chunks into vector representations
- **Vector Database Provider**: The backend storage system (e.g., Milvus)
- **Distance Metric**: How similarity is calculated (cosine, euclidean, etc.)
- **Collection Name**: A named collection where embeddings are stored

The vector store is initialized and ready to receive document chunks.

In [ ]:
from ai4rag.rag.embedding.llama_stack import LSEmbeddingModel, LSEmbeddingParams
from ai4rag.rag.vector_store.llama_stack import LSVectorStore

embedding_model_id = "{EMBEDDING_MODEL_ID}"
params = LSEmbeddingParams(**{EMBEDDING_PARAMS})

embedding_model = LSEmbeddingModel(client=client, model_id=embedding_model_id, params=params)

provider_id = "{PROVIDER_ID}"
distance_metric = "{DISTANCE_METRIC}"
collection_name = "{COLLECTION_NAME}"

ls_vectorstore = LSVectorStore(
    embedding_model=embedding_model,
    client=client,
    provider_id=provider_id,
    distance_metric=distance_metric,
    reuse_collection_name=collection_name,
)

### Upload chunks to vector store

This section processes each extracted markdown file by:
- Loading the document content with metadata
- Splitting it into chunks using the configured chunker
- Generating embeddings and uploading them to the vector store

Once complete, all document chunks are indexed and ready for semantic search queries.

In [ ]:
paths = list(Path("step_outputs/extracted_text").glob("*.md"))

for p in sorted(paths):
    document = Document(
        page_content=p.read_text(encoding="utf-8", errors="replace"),
        metadata={{"document_id": p.stem}},
    )

    chunked_documents = chunker.split_documents([document])
    ls_vectorstore.add_documents(chunked_documents)

### Retrieve chunks for sample question

This section demonstrates how to perform a semantic search query against the populated vector store. You can test retrieval by searching for relevant chunks based on a sample question.

In [ ]:
from pprint import pprint

sample_question = input()

results = ls_vectorstore.search(query=sample_question, k=5)
for result in results:
    if isinstance(result, tuple):
        pprint(result[0].model_dump(mode="python"), indent=4)
        continue
    pprint(result.model_dump(mode="python"), indent=4)

---

## Summary

This notebook successfully processed documents from S3 storage, extracted their text content using Docling, chunked the text into manageable pieces, and uploaded the embeddings to a vector store. The indexed documents are now ready for semantic search and retrieval in RAG applications.